# 01 - Section 1: Data exploration

**GeoAI LAB - Soil Texture from Spectra** | tasks 1.1 - 1.15

Requires `00_setup.ipynb` to have been run once (it builds `cache/*.parquet`).

Every figure is written to `figs/fig_1_<task>.png` and every table to
`tables/table_1_<task>.csv`, so each deliverable is findable by its task number.

## The one decision taken up front

`L400`-`L450` (11 wavelengths) are missing for **all 22 171 LUCAS.SSL rows** and present for
every other row: the XDS instrument LUCAS uses starts at 455 nm. This is a structural sensor
difference, not random missingness, and it is aligned exactly with the programme/instrument
confound recorded in step 0.

Two column groups are therefore defined, and used deliberately:

| group | size | used for |
|---|---|---|
| `SPEC` | 421 (400-2500 nm) | **plots**, via `nanmean`/`nanstd`, so the truncation stays visible |
| `SPEC410` | 410 (455-2500 nm) | **algebra**: PCA, correlations, and every model from section 4 on |

Dropping 11 noisy blue-end wavelengths costs little; dropping the LUCAS rows instead would
discard 55% of the data and one of the three programmes. Task 2.1 inherits this: band B1
(443 nm, 20 nm wide = 433-453 nm) lies entirely inside the missing block.

## Bootstrap

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "cache" / "dataset.parquet").exists():
    ROOT = ROOT.parent
assert (ROOT / "cache" / "dataset.parquet").exists(), "run 00_setup.ipynb first"

CACHE, FIGS, TABLES = ROOT / "cache", ROOT / "figs", ROOT / "tables"
FIGS.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)

dataset = pd.read_parquet(CACHE / "dataset.parquet")
soillab = pd.read_parquet(CACHE / "soillab.parquet")
soilsite = pd.read_parquet(CACHE / "soilsite.parquet")

SPEC = sorted((c for c in dataset.columns if re.fullmatch(r"L\d+", c)), key=lambda c: int(c[1:]))
WL = np.array([int(c[1:]) for c in SPEC])
META = [c for c in dataset.columns if c not in SPEC]
TARGETS = ["sand", "silt", "clay"]

PROGS = ["ICRAF.ISRIC", "KSSL.SSL", "LUCAS.SSL"]
PCOL = dict(zip(PROGS, ["#1b7f79", "#c1553b", "#3d5a99"]))

plt.rcParams.update({
    "figure.figsize": (9, 5), "figure.dpi": 110, "savefig.dpi": 150,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})


def save_fig(task, fig=None):
    path = FIGS / f"fig_{str(task).replace('.', '_')}.png"
    (fig or plt.gcf()).savefig(path)
    print(f"saved {path.relative_to(ROOT)}")
    return path


def save_table(task, df):
    path = TABLES / f"table_{str(task).replace('.', '_')}.csv"
    df.to_csv(path)
    print(f"saved {path.relative_to(ROOT)}")
    return path


print(dataset.shape, "|", len(SPEC), "spectral columns")

### The missing block, and `SPEC410`

Derived from the data rather than hard-coded, so it stays correct if the files change.

In [ ]:
nan_frac = np.isnan(dataset[SPEC].to_numpy()).mean(axis=0)

MISSING_WL = WL[nan_frac > 0]
SPEC410 = [c for c, f in zip(SPEC, nan_frac) if f == 0]
WL410 = np.array([int(c[1:]) for c in SPEC410])

# Two rows are flat -- reflectance == 1.0 at every wavelength. Failed measurements, not
# spectra. Reported properly in 1.2; excluded from the algebra here.
FLAT = np.nanstd(dataset[SPEC].to_numpy(), axis=1) == 0
VALID = ~FLAT

X = dataset.loc[VALID, SPEC410].to_numpy()   # NaN-free and non-degenerate, for all algebra
assert not np.isnan(X).any(), "SPEC410 still contains NaN"
assert (X.std(axis=0) > 0).all(), "a wavelength has zero variance"

print(f"missing wavelengths : {len(MISSING_WL)}  ({MISSING_WL.min()}-{MISSING_WL.max()} nm, "
      f"contiguous={np.all(np.diff(MISSING_WL) == 5)})")
print(f"missing in {nan_frac.max() * 100:.1f}% of rows -> ", end="")
print(dataset.loc[np.isnan(dataset["L400"].to_numpy()), "programme"].value_counts().to_dict())
print(f"SPEC    : {len(SPEC)} cols, {WL.min()}-{WL.max()} nm   (plots, nan-aware)")
print(f"SPEC410 : {len(SPEC410)} cols, {WL410.min()}-{WL410.max()} nm   (algebra, models)")
print(f"flat rows excluded from algebra : {int(FLAT.sum())}  -> X is {X.shape}  (see 1.2)")

## 1.1 - Load the file. Report its size and its columns.

**Deliverable:** shape + column list.

In [ ]:
print("FILE SHAPES")
for name, df in [("dataset", dataset), ("soillab", soillab), ("soilsite", soilsite)]:
    print(f"  {name:9s} {df.shape[0]:,} rows x {df.shape[1]:3d} cols")
print(f"  {'merged':9s} {len(dataset):,} rows x "
      f"{dataset.shape[1] + soillab.shape[1] + soilsite.shape[1] - 2:3d} cols  (on id)")

print("\nCOLUMNS OF `dataset` (the working table)")
print(f"  metadata ({len(META)}):")
for c in META:
    print(f"    {c:14s} {str(dataset[c].dtype):8s}  e.g. {dataset[c].iloc[0]!r}")
print(f"  spectra ({len(SPEC)}):")
print(f"    {SPEC[0]} .. {SPEC[-1]}, reflectance every 5 nm from {WL.min()} to {WL.max()} nm")

print("\nMEMORY")
print(f"  dataset in RAM: {dataset.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 1.2 - Check missing values, duplicates and value ranges.

**Deliverable:** 1 summary table.

The 421 spectral columns are folded into two rows - the missing block and the rest -
because 421 near-identical rows would hide the one thing that matters.

In [ ]:
rows = []
for c in META:
    s = dataset[c]
    numeric = pd.api.types.is_numeric_dtype(s)
    rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "n_missing": int(s.isna().sum()),
        "pct_missing": round(100 * s.isna().mean(), 3),
        "n_unique": int(s.nunique(dropna=True)),
        "min": round(float(s.min()), 4) if numeric else "",
        "max": round(float(s.max()), 4) if numeric else "",
    })


def spec_row(cols, label):
    A = dataset[cols].to_numpy()
    return {
        "column": label,
        "dtype": "float64",
        "n_missing": int(np.isnan(A).sum()),
        "pct_missing": round(100 * np.isnan(A).mean(), 3),
        "n_unique": "",
        "min": round(float(np.nanmin(A)), 4),
        "max": round(float(np.nanmax(A)), 4),
    }


miss_cols = [c for c in SPEC if c not in set(SPEC410)]
rows.append(spec_row(miss_cols, f"L400-L450 ({len(miss_cols)} cols)"))
rows.append(spec_row(SPEC410, f"L455-L2500 ({len(SPEC410)} cols)"))

summary = pd.DataFrame(rows).set_index("column")
save_table("1.2", summary)
summary

### Duplicates and range checks

These do not fit in the table above but belong to the same task.

In [ ]:
print("DUPLICATES")
print(f"  duplicate id                     : {dataset['id'].duplicated().sum()}")
print(f"  duplicate spectra (421 cols)     : {dataset.duplicated(subset=SPEC).sum()}")
print(f"  duplicate spectra + targets      : {dataset.duplicated(subset=SPEC + TARGETS).sum()}")
print("    -> identical spectra carrying DIFFERENT texture values. Small, but it caps the")
print("       achievable score and is a mild leakage risk if a pair straddles the split.")

print("\nRANGE CHECKS")
A = dataset[SPEC].to_numpy()
print(f"  reflectance min / max            : {np.nanmin(A):.4f} / {np.nanmax(A):.4f}")
print(f"  values < 0                       : {int((A < 0).sum())}")
print(f"  values > 1                       : {int((A > 1).sum())}")
print(f"  values == 1.0 exactly            : {int((A == 1.0).sum())}  "
      f"in {int((A == 1.0).any(axis=1).sum())} rows")
print(f"  flat rows (zero variance)        : {int(FLAT.sum())}")
print(dataset.loc[FLAT, ["id", "programme", "instrument"] + TARGETS].to_string(index=False))
print("    -> every saturated value lives in these rows: reflectance is exactly 1.0 at all")
print("       421 wavelengths. Failed measurements, not spectra. They carry perfectly valid")
print("       texture values, so a target-based filter would never catch them. Excluded from")
print("       PCA and correlation below; drop them before modelling in section 4.")
print(f"  n_scans range                    : {dataset['n_scans'].min():.0f} - "
      f"{dataset['n_scans'].max():.0f}")
print(f"  prof_bas < prof_haut             : {int((dataset['prof_bas'] < dataset['prof_haut']).sum())}")
print(f"  zero-thickness layers            : {int((dataset['prof_bas'] == dataset['prof_haut']).sum())}")
print(f"  missing lon / lat                : {int(dataset['lon'].isna().sum())} rows "
      f"({100 * dataset['lon'].isna().mean():.1f}%)")
print(f"  targets outside [0, 100]         : "
      f"{int(((dataset[TARGETS] < 0) | (dataset[TARGETS] > 100)).sum().sum())}")

## 1.3 - Plot 20 spectra on the same axes.

**Deliverable:** 1 figure.

Coloured by programme rather than arbitrarily: the LUCAS curves starting at 455 nm are the
visual confirmation of what 1.2 reported numerically.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(dataset), 20, replace=False)

fig, ax = plt.subplots()
seen = set()
for i in idx:
    prog = dataset["programme"].iloc[i]
    ax.plot(WL, dataset[SPEC].iloc[i].to_numpy(), lw=0.9, alpha=0.85,
            color=PCOL[prog], label=prog if prog not in seen else None)
    seen.add(prog)
ax.axvspan(395, 452, color="0.85", zorder=0, label="missing for LUCAS")
ax.set(xlabel="wavelength (nm)", ylabel="reflectance",
       title="1.3 - Twenty randomly drawn spectra (seed 0)")
ax.legend(fontsize=8)
save_fig("1.3")
plt.show()

print(dataset["programme"].iloc[idx].value_counts().to_dict())

## 1.4 - Plot the mean spectrum with +/-1 standard deviation.

**Deliverable:** 1 figure.

`nanmean`/`nanstd` over `SPEC`, so the 400-450 block is computed from the 45% of rows that
have it. The shaded span marks it - otherwise the step at 455 nm looks like a data error.

In [ ]:
A = dataset[SPEC].to_numpy()
mu, sd = np.nanmean(A, axis=0), np.nanstd(A, axis=0)

fig, ax = plt.subplots()
ax.fill_between(WL, mu - sd, mu + sd, alpha=0.25, color="#3d5a99", label="+/-1 SD")
ax.plot(WL, mu, color="#1a2b4f", lw=1.6, label="mean")
ax.axvspan(395, 452, color="0.85", zorder=0, label="missing for LUCAS (n=18 364 here)")
for w, txt in [(1400, "1400"), (1900, "1900"), (2200, "2200")]:
    ax.axvline(w, color="0.55", ls=":", lw=1)
    ax.annotate(txt, (w, ax.get_ylim()[1]), fontsize=8, ha="center", va="top", color="0.4")
ax.set(xlabel="wavelength (nm)", ylabel="reflectance",
       title="1.4 - Mean spectrum +/-1 SD  (dotted: water / clay-mineral absorption)")
ax.legend(fontsize=8, loc="lower right")
save_fig("1.4")
plt.show()

## 1.5 - Plot the mean spectrum, one curve per programme.

**Deliverable:** 1 figure.

Two things to read off this figure, both of which matter for section 5: the LUCAS curve
starts at 455 nm, and the three curves are offset from one another - programme and
instrument are the same variable, so an offset here is *both* a soil difference and an
instrument difference, with no way to separate them.

In [ ]:
means = dataset.groupby("programme")[SPEC].mean()

fig, ax = plt.subplots()
for prog in PROGS:
    ax.plot(WL, means.loc[prog].to_numpy(), lw=1.5, color=PCOL[prog],
            label=f"{prog}  (n={int((dataset['programme'] == prog).sum()):,})")
ax.axvspan(395, 452, color="0.85", zorder=0)
ax.set(xlabel="wavelength (nm)", ylabel="mean reflectance",
       title="1.5 - Mean spectrum per programme (= per instrument)")
ax.legend(fontsize=8)
save_fig("1.5")
plt.show()

print("programme x instrument (perfectly confounded):")
print(pd.crosstab(dataset["programme"], dataset["instrument"]).to_string())

## 1.6 - Run a PCA on the spectral columns only.

**Deliverable:** number of components reaching 95% of variance.

Two choices, both stated rather than defaulted:

- **`SPEC410`, not `SPEC`** - PCA cannot accept NaN, and dropping 11 columns is cheaper than
  dropping 22 171 rows.
- **Mean-centred, not standardised** - all 410 columns are the same physical quantity in the
  same unit, so standardising would inflate the noisy blue end into a false principal
  direction. The scaled count is printed underneath for comparison.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca = PCA().fit(X)                       # PCA centres internally, does not scale
cum = np.cumsum(pca.explained_variance_ratio_)
n95 = int(np.searchsorted(cum, 0.95) + 1)

print(f"ANSWER 1.6 : {n95} component(s) reach 95% of variance "
      f"({len(SPEC410)} spectral columns in)")
print(f"  explained by PC1..PC5 : "
      f"{np.round(pca.explained_variance_ratio_[:5] * 100, 2).tolist()} %")
for k in (0.90, 0.95, 0.99, 0.999):
    print(f"  {k * 100:5.1f}% of variance -> {int(np.searchsorted(cum, k) + 1):3d} components")

n95_scaled = int(np.searchsorted(
    np.cumsum(PCA().fit(StandardScaler().fit_transform(X)).explained_variance_ratio_), 0.95) + 1)
print(f"\n  for comparison, if standardised instead: {n95_scaled} components")

print("\n  per programme:")
for p in PROGS:
    m = ((dataset["programme"] == p).to_numpy()) & VALID
    e = PCA().fit(dataset.loc[m, SPEC410].to_numpy()).explained_variance_ratio_
    print(f"    {p:12s} {int(np.searchsorted(np.cumsum(e), 0.95) + 1):2d} components at 95%"
          f"   (PC1 alone = {e[0] * 100:.1f}%)")

print("\n  PC1's loadings all share one sign (see the right-hand panel): PC1 is overall")
print("  brightness/albedo, not a texture feature. That single number is why 1 component")
print("  clears 95% -- soil spectra differ far more in level than in shape.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(np.arange(1, 31), cum[:30] * 100, marker="o", ms=3, color="#1a2b4f")
ax1.axhline(95, color="#c1553b", ls="--", lw=1)
ax1.axvline(n95, color="#c1553b", ls="--", lw=1)
ax1.annotate(f"{n95} components", (n95, 60), fontsize=9, color="#c1553b",
             xytext=(n95 + 2, 55))
ax1.set(xlabel="number of components", ylabel="cumulative variance (%)",
        title=f"1.6 - {n95} components reach 95%", ylim=(50, 101))

for k in range(3):
    ax2.plot(WL410, pca.components_[k], lw=1.2,
             label=f"PC{k + 1} ({pca.explained_variance_ratio_[k] * 100:.1f}%)")
ax2.axhline(0, color="0.7", lw=0.8)
ax2.set(xlabel="wavelength (nm)", ylabel="loading", title="First three loadings")
ax2.legend(fontsize=8)
save_fig("1.6")
plt.show()

## 1.7 - Compute the correlation between two neighbouring wavelengths.

**Deliverable:** 1 number.

The pair is taken **outside** 400-450 nm: `L400` vs `L405` would silently be computed on
45% of the rows. The contrast is printed to show why that matters.

In [ ]:
r_1000 = float(np.corrcoef(dataset["L1000"], dataset["L1005"])[0, 1])
print(f"ANSWER 1.7 : r(L1000, L1005) = {r_1000:.6f}   (n = {len(dataset):,})")

sub = dataset[["L400", "L405"]].dropna()
print(f"\n  the trap: r(L400, L405) = "
      f"{float(np.corrcoef(sub['L400'], sub['L405'])[0, 1]):.6f} "
      f"but only n = {len(sub):,} -- a different population, not a comparable number")

adj = np.array([np.corrcoef(X[:, i], X[:, i + 1])[0, 1] for i in range(X.shape[1] - 1)])
print(f"\n  across all 409 adjacent pairs in SPEC410: "
      f"min {adj.min():.5f}  median {np.median(adj):.5f}  max {adj.max():.5f}")
print(f"  -> {n95} components carry 95% of 410 columns because neighbours are ~identical.")
print("     This is the justification for the whole of section 2.")

## 1.8 - Plot the distribution of sand, silt and clay.

**Deliverable:** 1 figure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, t, col in zip(axes, TARGETS, ["#c98b3b", "#4a8c6f", "#8c4a6f"]):
    v = dataset[t]
    ax.hist(v, bins=60, color=col, alpha=0.85)
    ax.axvline(v.mean(), color="k", ls="--", lw=1, label=f"mean {v.mean():.1f}")
    ax.axvline(v.median(), color="k", ls=":", lw=1, label=f"median {v.median():.1f}")
    ax.set(xlabel=f"{t} (%)", title=f"{t}: {v.min():.0f} - {v.max():.1f}%")
    ax.legend(fontsize=8)
axes[0].set_ylabel("count")
fig.suptitle("1.8 - Distribution of the three texture fractions", y=1.02)
save_fig("1.8")
plt.show()

print(dataset[TARGETS].describe().T.round(2).to_string())
print("\nskew:", dataset[TARGETS].skew().round(3).to_dict())

## 1.9 - Compute sand+silt+clay for every sample.

**Deliverable:** min, mean, max.

In [ ]:
s = dataset[TARGETS].sum(axis=1)
print(f"ANSWER 1.9 : min {s.min():.6f}   mean {s.mean():.6f}   max {s.max():.6f}")
print(f"  spread = {s.max() - s.min():.6f} -> rounding noise only")
print(f"  rows more than 0.01 from 100 : {int((s.sub(100).abs() > 0.01).sum())}")
print("""
Two consequences, both load-bearing later:

  1. The fractions are already closed, and closed to 100 -- not to 1. Task 4.3's
     "further than 2% from 1" therefore means 2 points away from 100.
  2. Only two of the three targets are independent: sand = 100 - silt - clay. Any gap
     measured in 4.3 is created purely by fitting the three separately; nothing in the
     data produces it.
""")

## 1.10 - Place the samples in the soil texture triangle.

**Deliverable:** 1 figure.

Neither `mpltern` nor `python-ternary` is installed, and neither is needed: with the
fractions already summing to 100, a barycentric transform is two lines. 40 535 points as a
plain scatter is a solid blob, so density (hexbin, log counts) is plotted instead.

In [ ]:
sand, silt, clay = (dataset[t].to_numpy() for t in TARGETS)
x = 0.5 * (2 * silt + clay) / 100.0
y = (np.sqrt(3) / 2) * clay / 100.0

fig, ax = plt.subplots(figsize=(7.5, 6.8))
hb = ax.hexbin(x, y, gridsize=60, bins="log", cmap="magma_r", mincnt=1)
tri = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3) / 2], [0, 0]])
ax.plot(tri[:, 0], tri[:, 1], color="0.25", lw=1.2)
for frac in (0.2, 0.4, 0.6, 0.8):                      # iso-clay guides
    ax.plot([frac * 0.5, 1 - frac * 0.5],
            [frac * np.sqrt(3) / 2, frac * np.sqrt(3) / 2], color="0.8", lw=0.6, zorder=0)
ax.annotate("100% sand", (0, -0.03), ha="center", fontsize=9)
ax.annotate("100% silt", (1, -0.03), ha="center", fontsize=9)
ax.annotate("100% clay", (0.5, np.sqrt(3) / 2 + 0.02), ha="center", fontsize=9)
fig.colorbar(hb, ax=ax, shrink=0.7, label="samples per cell (log)")
ax.set(title=f"1.10 - Texture triangle, all {len(dataset):,} samples", xticks=[], yticks=[])
ax.set_aspect("equal")
ax.grid(False)
ax.set_axis_off()
save_fig("1.10")
plt.show()

## 1.11 - Count the samples per texture class.

**Deliverable:** 1 table.

`classe_usda` already exists and has no missing values, so it is used directly rather than
re-deriving the USDA boundaries.

In [ ]:
counts = dataset["classe_usda"].value_counts().to_frame("n")
counts["pct"] = (100 * counts["n"] / len(dataset)).round(2)
save_table("1.11", counts)

print(f"{counts.shape[0]} classes, 0 missing")
print(f"imbalance: {counts['n'].max():,} ({counts.index[0]}) vs {counts['n'].min():,} "
      f"({counts.index[-1]}) = {counts['n'].max() / counts['n'].min():.0f}x")
counts

## 1.12 - Map the samples using lon / lat.

**Deliverable:** 1 figure.

This is the figure that completes the confound: programme is not only a laboratory and an
instrument, it is also a **region**. Task 5.1's leave-one-programme-out is therefore
simultaneously leave-one-instrument-out and leave-one-continent-out.

In [ ]:
geo = dataset.dropna(subset=["lon", "lat"])
print(f"dropped {len(dataset) - len(geo)} rows with missing coordinates "
      f"({100 * (1 - len(geo) / len(dataset)):.1f}%)")

fig, ax = plt.subplots(figsize=(12, 5.6))
for prog in PROGS:
    g = geo[geo["programme"] == prog]
    ax.scatter(g["lon"], g["lat"], s=2, alpha=0.35, color=PCOL[prog],
               label=f"{prog}  (n={len(g):,})", linewidths=0)
ax.set(xlabel="longitude", ylabel="latitude", xlim=(-180, 180), ylim=(-90, 90),
       xticks=range(-180, 181, 30), yticks=range(-90, 91, 30),
       title="1.12 - Sample locations (no basemap: cartopy not installed)")
ax.set_aspect("equal")
leg = ax.legend(fontsize=8, markerscale=5, loc="lower left")
for h in leg.legend_handles:
    h.set_alpha(1)
save_fig("1.12")
plt.show()

print("\nmissing coordinates by programme:")
print(dataset[dataset["lon"].isna()]["programme"].value_counts().to_dict())

## 1.13 - Count samples per programme, and box-plot clay per programme.

**Deliverable:** 1 table + 1 figure.

In [ ]:
prog_tbl = dataset.groupby("programme").agg(
    n=("id", "size"),
    instrument=("instrument", lambda s: s.unique()[0]),
    clay_mean=("clay", "mean"),
    clay_median=("clay", "median"),
    clay_min=("clay", "min"),
    clay_max=("clay", "max"),
    clay_p95=("clay", lambda s: s.quantile(0.95)),
).round(2)
prog_tbl["pct"] = (100 * prog_tbl["n"] / len(dataset)).round(1)
save_table("1.13", prog_tbl)
prog_tbl

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4),
                               gridspec_kw={"width_ratios": [1, 1.6]})
ax1.bar(range(len(PROGS)), [int((dataset["programme"] == p).sum()) for p in PROGS],
        color=[PCOL[p] for p in PROGS])
ax1.set(xticks=range(len(PROGS)), xticklabels=[p.replace(".", "\n") for p in PROGS],
        ylabel="samples", title="1.13 - Samples per programme")

data = [dataset.loc[dataset["programme"] == p, "clay"].to_numpy() for p in PROGS]
bp = ax2.boxplot(data, tick_labels=[p.replace(".", "\n") for p in PROGS],
                 patch_artist=True, showfliers=True,
                 flierprops=dict(marker=".", ms=2, alpha=0.2))
for patch, p in zip(bp["boxes"], PROGS):
    patch.set_facecolor(PCOL[p])
    patch.set_alpha(0.65)
for med in bp["medians"]:
    med.set_color("k")
ax2.set(ylabel="clay (%)", title="Clay per programme")
save_fig("1.13")
plt.show()

## 1.14 - Plot the histogram of the sampling depth.

**Deliverable:** 1 figure.

The columns are `prof_haut` / `prof_bas` (the PDF calls them `depth_top` /
`depth_bottom`); they are bit-identical to `soilsite`'s `layer.upper/lower.depth_usda_cm`.

In [ ]:
top = dataset["prof_haut"]
thick = dataset["prof_bas"] - dataset["prof_haut"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(top, bins=60, color="#6b5b95")
ax1.axvline(top.median(), color="k", ls="--", lw=1, label=f"median {top.median():.0f} cm")
ax1.set(xlabel="prof_haut - top of layer (cm)", ylabel="count",
        title=f"1.14 - Sampling depth (top)   0-{top.max():.0f} cm", yscale="log")
ax1.legend(fontsize=8)

ax2.hist(thick, bins=60, color="#907b62")
ax2.axvline(thick.median(), color="k", ls="--", lw=1, label=f"median {thick.median():.0f} cm")
ax2.set(xlabel="layer thickness (cm)", title="Layer thickness", yscale="log")
ax2.legend(fontsize=8)
save_fig("1.14")
plt.show()

print(dataset[["prof_haut", "prof_bas"]].describe().T.round(1).to_string())
print(f"\nsurface layers (prof_haut == 0): {int((top == 0).sum()):,} "
      f"({100 * (top == 0).mean():.1f}%)")
print("depth by programme (median top / median thickness):")
for p in PROGS:
    g = dataset[dataset["programme"] == p]
    print(f"  {p:12s} {g['prof_haut'].median():5.0f} cm  /  "
          f"{(g['prof_bas'] - g['prof_haut']).median():4.0f} cm")

## 1.15 - Correlation of every wavelength with clay, silt and sand.

**Deliverable:** 1 figure + the peak wavelength of each target, written down.

Computed as one matrix product on `SPEC410` rather than a 1 230-iteration loop. The peak is
taken on **|r|**, because clay correlates *negatively* with reflectance across most of the
range - `argmax(r)` would return the wrong wavelength.

Tasks 3.1 and 3.2 compare directly against this figure, so it is the reference point for
everything that follows.

In [ ]:
def corr_curves(mask):
    """Pearson r of every SPEC410 wavelength against each target, as one matrix product."""
    Xa = dataset.loc[mask, SPEC410].to_numpy()
    Ya = dataset.loc[mask, TARGETS].to_numpy()
    Xz = (Xa - Xa.mean(axis=0)) / Xa.std(axis=0)
    Yz = (Ya - Ya.mean(axis=0)) / Ya.std(axis=0)
    return Xz.T @ Yz / len(Xa)                          # (410, 3)


R = corr_curves(VALID)

peaks = {}
for k, t in enumerate(TARGETS):
    j = int(np.argmax(np.abs(R[:, k])))
    peaks[t] = (int(WL410[j]), float(R[j, k]))

print("ANSWER 1.15 - peak |r| wavelength per target (SPEC410, n = "
      f"{len(dataset):,})\n")
for t, (w, r) in peaks.items():
    print(f"  {t:5s} : {w} nm   r = {r:+.4f}")
print("\n  r range per target:")
for k, t in enumerate(TARGETS):
    print(f"  {t:5s} : {R[:, k].min():+.4f} .. {R[:, k].max():+.4f}")
print(f"\n  corr(clay curve, sand curve) = "
      f"{np.corrcoef(R[:, 2], R[:, 0])[0, 1]:+.4f}  -> near-mirror images, as expected from 1.9")

R_prog = {p: corr_curves(((dataset["programme"] == p).to_numpy()) & VALID) for p in PROGS}

print("\n\nWHY THE POOLED VALUES ARE SO LOW - the same curves, one programme at a time:\n")
print(f"  {'programme':13s} {'n':>7s}   " + "   ".join(f"{t:>18s}" for t in TARGETS))
for p in ["POOLED"] + PROGS:
    Rp = R if p == "POOLED" else R_prog[p]
    npts = int(VALID.sum()) if p == "POOLED" else int((((dataset['programme'] == p).to_numpy()) & VALID).sum())
    cellsx = []
    for k in range(3):
        j = int(np.argmax(np.abs(Rp[:, k])))
        cellsx.append(f"{int(WL410[j]):>5d} nm r={Rp[j, k]:+.3f}")
    print(f"  {p:13s} {npts:7,}   " + "   ".join(cellsx))

print("\n  mean reflectance (455-2500 nm) per programme:")
for p in PROGS:
    m = ((dataset["programme"] == p).to_numpy()) & VALID
    print(f"    {p:12s} {dataset.loc[m, SPEC410].to_numpy().mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for k, (t, col) in enumerate(zip(TARGETS, ["#c98b3b", "#4a8c6f", "#8c4a6f"])):
    ax.plot(WL410, R[:, k], lw=1.5, color=col, label=t)
    w, r = peaks[t]
    ax.plot([w], [r], marker="o", ms=6, color=col, mec="k", mew=0.8, zorder=5)
    ax.annotate(f"{t} peak {w} nm (r={r:+.2f})", (w, r), fontsize=8, color=col,
                xytext=(6, 10 if r > 0 else -14), textcoords="offset points")
ax.axhline(0, color="0.6", lw=0.9)
for w in (1400, 1900, 2200):
    ax.axvline(w, color="0.75", ls=":", lw=1)
ax.set(xlabel="wavelength (nm)", ylabel="Pearson r with target",
       title="1.15 - Correlation of each wavelength with each texture fraction")
ax.legend(fontsize=9)
save_fig("1.15")
plt.show()

### 1.15b - the same curves per programme (supporting figure)

The pooled peaks above are weak: |r| = 0.07 for sand. That is not an absence of signal, it
is **dilution by the confound**. Mean reflectance is 0.65 for LUCAS against 0.40 for the
other two, so pooling makes the largest axis of variation "which instrument measured this",
and the within-programme relationship between reflectance and texture is averaged away.

Split by programme, the same computation gives correlations up to |r| = 0.61 - and the
peaks land at **different wavelengths, with opposite signs** (clay peaks near 2475 nm and
negative for ICRAF, near 605 nm and positive for LUCAS). Three different relationships are
being averaged into one weak curve.

This is the single most important figure for section 3: it says up front that a model
trained across programmes and evaluated within them is not measuring the same thing as one
evaluated across them, which is exactly what task 5.1 will confront.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, p in zip(axes, PROGS):
    Rp = R_prog[p]
    for k, (t, col) in enumerate(zip(TARGETS, ["#c98b3b", "#4a8c6f", "#8c4a6f"])):
        ax.plot(WL410, Rp[:, k], lw=1.3, color=col, label=t)
        j = int(np.argmax(np.abs(Rp[:, k])))
        ax.plot([WL410[j]], [Rp[j, k]], marker="o", ms=5, color=col, mec="k", mew=0.7)
    ax.axhline(0, color="0.6", lw=0.9)
    ax.plot(WL410, R[:, 2], lw=1.0, ls="--", color="0.5", label="clay, pooled")
    n_p = int((((dataset["programme"] == p).to_numpy()) & VALID).sum())
    ax.set(xlabel="wavelength (nm)", title=f"{p}  (n={n_p:,})", ylim=(-0.75, 0.75))
axes[0].set_ylabel("Pearson r with target")
axes[0].legend(fontsize=8, ncol=2)
fig.suptitle("1.15b - Correlation curves computed within each programme "
             "(dashed grey = pooled clay curve, for reference)", y=1.03)
save_fig("1.15b")
plt.show()

## Section 1 - answers, and what carries forward

Transcribed from the executed run. The written values are the deliverables for 1.6, 1.7,
1.9 and 1.15.

| Task | Answer |
|---|---|
| **1.1** | `dataset` 40 535 x 433 (12 metadata + 421 spectra); `soillab` x 58; `soilsite` x 36 |
| **1.6** | **1 component** reaches 95% of variance (PC1 alone = 95.43%). 3 components for 99%, 6 for 99.9% |
| **1.7** | **r(L1000, L1005) = 0.999944**; across all 409 adjacent pairs, median 0.99999 |
| **1.9** | min **99.9999**, mean **100.0000**, max **100.0001** - already closed, to 100 |
| **1.11** | 12 classes, 0 missing. Silt loam 7 847 -> Sandy clay 137 = **57x imbalance** |
| **1.15** | pooled peak \|r\|: **sand 2500 nm** (r = +0.074), **silt 2170 nm** (r = +0.155), **clay 2500 nm** (r = -0.319) |

### 1.15 per programme - the number that matters more than the pooled one

| programme | n | sand | silt | clay |
|---|---|---|---|---|
| pooled | 40 533 | 2500 nm, +0.074 | 2170 nm, +0.155 | 2500 nm, -0.319 |
| ICRAF.ISRIC | 3 636 | 1910 nm, +0.359 | 2205 nm, +0.198 | **2475 nm, -0.606** |
| KSSL.SSL | 14 726 | 1275 nm, -0.060 | 2500 nm, +0.199 | 2500 nm, -0.361 |
| LUCAS.SSL | 22 171 | 605 nm, -0.369 | 625 nm, +0.358 | **605 nm, +0.237** |

Clay's peak correlation is **negative at 2475 nm** for ICRAF but **positive at 605 nm** for
LUCAS. Not a weaker version of the same relationship - a different one.

## What carries into sections 2-6

1. **`SPEC410` (455-2500 nm) is the modelling grid.** 400-450 nm exists only for KSSL and
   ICRAF. Task 2.1 inherits this: band **B1 (433-453 nm) is uncomputable for LUCAS** - decide
   in 2.1 whether to drop B1 for everyone or leave it NaN for 55% of rows, and justify it in
   2.5. The first ~4 of the 12 nm hyperspectral bands have the same problem.
2. **Drop the 2 flat rows before modelling** (`np.nanstd(spectra, axis=1) == 0`). Reflectance
   is exactly 1.0 at all 421 wavelengths; their texture values are valid, so nothing else
   catches them.
3. **1 component carries 95% of 410 columns, and PC1 is brightness.** Per programme it takes
   2-3 components - pooling inflates the apparent redundancy. Neighbouring wavelengths
   correlate at 0.99999. This is the justification for section 2: resampling to 12 or 175
   bands throws away far less than the column count suggests.
4. **The confound is threefold.** programme = instrument (step 0) = region (1.12), plus
   different depth regimes (1.14: LUCAS median top 0 cm, ICRAF 40 cm) and different albedo
   (0.65 vs 0.40). Task 5.1's leave-one-programme-out varies all four at once; report the
   drop as such, not as "domain shift".
5. **Targets are on 0-100 and already closed.** Task 4.3's threshold means 2 points from 100.
6. **11 duplicate spectra pairs** carry different texture values - a ceiling on any score,
   and a mild leakage risk if a pair straddles the section 4 split.

**Next:** section 2, sensor simulation (tasks 2.1-2.5).